# CSE457 — BÁO CÁO THỰC HÀNH LAB 4
## TỔNG HỢP TIẾNG NÓI (TEXT-TO-SPEECH SYNTHESIS)
**Sinh viên thực hiện:** Lê Văn Chương  
**Mã số sinh viên (MSSV):** 2351260643  
**Bộ môn:** Trí tuệ Nhân tạo — Khoa Công nghệ Thông tin, Trường Đại học Thủy lợi  
**Học phần:** CSE457 — Xử lý Âm thanh và Tiếng nói  

---
### Mục tiêu bài Lab:
1. Nắm vững kiến trúc Text-to-Speech (TTS): Text Analysis -> Phonetic Analysis -> Prosody -> Synthesis Backend -> Waveform.
2. Xây dựng Text Normalization và Speech Inventory miền hẹp ('Thông báo xếp hàng').
3. Thực hiện Concatenative TTS baseline và tối ưu hóa bằng **Viterbi Unit Selection** kết hợp **Boundary Cross-fade Smoothing**.
4. Phân tích ngữ điệu (Prosody) và mô hình **Source-Filter**: Đo F0, Time-stretch WSOLA, Pitch shift semitones.
5. Nghiên cứu tổng hợp tham số thống kê **HMM-based Speech Synthesis**: Quỹ đạo tham số với ràng buộc động MLPG và bộ giải mã nguồn-lọc Formant Vocoder.
6. Thực nghiệm **Neural TTS** học sâu (mô hình pretrained) và đo độ trễ Latency, Real-Time Factor (RTF).
7. Đánh giá toàn diện 3 phương pháp qua trắc nghiệm thính giác (Listening test) theo chuẩn MOS (Intelligibility, Naturalness, Continuity, Latency, Storage Footprint).


In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import librosa.display
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import Audio, display

# Import backend modules
from hmm_tts_backend import synthesize_hmm
from neural_tts_backend import synthesize

SR = 16000
DATA_DIR = "lab4_data"
UNITS_DIR = os.path.join(DATA_DIR, "units")
METADATA_PATH = os.path.join(DATA_DIR, "metadata.csv")
OUTPUT_DIR = "audio_output"
FIGURES_DIR = "figures"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print("Khởi tạo môi trường hoàn tất. Tần số lấy mẫu SR =", SR)


## Phần A: Text Normalization & Speech Inventory
Trong hệ thống TTS, khối **Text Analysis** (Front-end) đóng vai trò quyết định trong việc biến chuỗi ký tự thô $T$ thành chuỗi token từ ngữ chuẩn hóa $W = (w_1, w_2, \dots, w_K)$.
- Với miền hẹp 'Thông báo xếp hàng':
  - Chuyển toàn bộ ký tự sang chữ thường (lowercase).
  - Loại bỏ các dấu câu ngăn cách (dấu chấm, phẩy, chấm phẩy).
  - Chuỗi số nguyên (ví dụ: '23', '5') được ánh xạ thành từng chữ số độc lập: '23' -> ['hai', 'ba'], '5' -> ['năm'].


In [ ]:
DIGIT_MAP = {
    "0": "không", "1": "một", "2": "hai", "3": "ba", "4": "bốn",
    "5": "năm", "6": "sáu", "7": "bảy", "8": "tám", "9": "chín"
}

VI_TO_FILENAME = {
    "mời": "moi", "khách": "khach", "hàng": "hang", "số": "so",
    "không": "khong", "một": "mot", "hai": "hai", "ba": "ba",
    "bốn": "bon", "năm": "nam", "sáu": "sau", "bảy": "bay",
    "tám": "tam", "chín": "chin", "đến": "den", "quầy": "quay"
}

def normalize_queue_text(text: str) -> list:
    cleaned = text.lower().replace(".", "").replace(",", "").replace(";", "").replace(":", "")
    tokens = []
    for tok in cleaned.split():
        if tok.isdigit():
            tokens.extend(DIGIT_MAP[d] for d in tok if d in DIGIT_MAP)
        else:
            tokens.append(tok)
    return tokens

test_sent_1 = "Mời khách hàng số 23 đến quầy 5."
tokens_1 = normalize_queue_text(test_sent_1)
print(f"Câu gốc: '{test_sent_1}'")
print(f"Chuỗi tokens chuẩn hóa: {tokens_1}")


In [ ]:
# Đọc và hiển thị Metadata của Inventory
df_meta = pd.read_csv(METADATA_PATH)
print(f"Tổng số unit trong kho âm thanh: {len(df_meta)}")
print(f"Số từ vựng phân biệt: {df_meta['label'].nunique()}")
display(df_meta.head(10))

# Trực quan hóa thống kê Inventory (Duration và RMS)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df_meta['duration'], bins=10, color='#2b5c8f', edgecolor='black', alpha=0.85)
axes[0].axvline(df_meta['duration'].mean(), color='red', linestyle='--', linewidth=1.5, label=f"Mean: {df_meta['duration'].mean():.3f}s")
axes[0].set_title("Phân bố thời lượng Unit (Duration)", fontweight='bold')
axes[0].set_xlabel("Thời lượng (giây)")
axes[0].set_ylabel("Số lượng unit")
axes[0].legend()
axes[0].grid(True, linestyle=':', alpha=0.6)

axes[1].hist(df_meta['rms'], bins=10, color='#2ca02c', edgecolor='black', alpha=0.85)
axes[1].axvline(df_meta['rms'].mean(), color='darkred', linestyle='--', linewidth=1.5, label=f"Mean RMS: {df_meta['rms'].mean():.3f}")
axes[1].set_title("Phân bố năng lượng RMS của Unit", fontweight='bold')
axes[1].set_xlabel("RMS")
axes[1].set_ylabel("Số lượng unit")
axes[1].legend()
axes[1].grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()


## Phần B: Baseline Concatenative Synthesis
- Phương pháp ghép nối trực tiếp (Raw Concatenation):
  - Lấy một bản thu cố định (candidate_01) cho mỗi từ.
  - Ghép nối thẳng các mảng tín hiệu liên tiếp nhau mà không dùng làm trơn biên (cross-fade = 0 ms).
  - Đánh dấu chính xác vị trí mẫu biên nối để phân tích hiện tượng click và gián đoạn phổ.


In [ ]:
def load_unit(path: str, sr: int = SR) -> np.ndarray:
    x, fs = sf.read(path, dtype="float32")
    if x.ndim > 1: x = x.mean(axis=1)
    if fs != sr: x = librosa.resample(x, orig_sr=fs, target_sr=sr)
    return x.astype(np.float32)

def calc_rms(x: np.ndarray) -> float:
    return float(np.sqrt(np.mean(x * x) + 1e-12))

def match_rms(x: np.ndarray, target: float = 0.08) -> np.ndarray:
    r = calc_rms(x)
    return np.clip(x * (target / r), -0.99, 0.99) if r > 1e-6 else x

def synthesize_raw_concat(tokens: list, candidate_id: str = "01"):
    parts = []
    boundaries = [0]
    curr_s = 0
    for tok in tokens:
        p = os.path.join(UNITS_DIR, f"{VI_TO_FILENAME[tok]}_{candidate_id}.wav")
        x = match_rms(load_unit(p, SR))
        parts.append(x)
        curr_s += len(x)
        boundaries.append(curr_s)
    return np.concatenate(parts), boundaries

y_raw, b_raw = synthesize_raw_concat(tokens_1, candidate_id="01")
sf.write(os.path.join(OUTPUT_DIR, "raw_concat.wav"), y_raw, SR)
print(f"Đã tạo raw_concat.wav, độ dài: {len(y_raw)/SR:.2f}s")
display(Audio(y_raw, rate=SR))


## Phần C: Unit Selection (Viterbi DP) & Boundary Smoothing
Khi mỗi từ vựng có nhiều candidate ($K \ge 2$), bài toán đặt ra là chọn ra chuỗi candidate $\theta^* = (\theta_1, \theta_2, \dots, \theta_N)$ sao cho tối thiểu hóa tổng chi phí:
$$\theta^* = \arg\min_{\theta_1 \dots \theta_N} \left[ \sum_{j=1}^N C_{target}(\theta_j, t_j) + \lambda \sum_{j=1}^{N-1} C_{join}(\theta_j, \theta_{j+1}) \right]$$
Trong đó:
- **Target Cost** $C_{target}$: Đo độ lệch về duration và RMS so với mục tiêu chuẩn.
- **Join Cost** $C_{join}$: Đo mức độ không tương thích tại vùng biên nối 25ms giữa 2 unit kế tiếp thông qua khoảng cách phổ Log-magnitude Spectral Distance:
$$D_{spec} = \frac{1}{K} \sum_{k=1}^K [L_{left}[k] - L_{right}[k]]^2 + w_e |RMS_{left} - RMS_{right}|$$
- **Boundary Cross-fade**: Trộn mượt 2 unit tại biên $N$ mẫu với hàm dốc tuyến tính $\alpha[n]$:
$$y[n] = (1 - \alpha[n]) x_1[n] + \alpha[n] x_2[n], \quad 0 \le n < N$$


In [ ]:
def crossfade(a: np.ndarray, b: np.ndarray, n: int) -> np.ndarray:
    n = min(n, len(a), len(b))
    if n == 0: return np.concatenate([a, b])
    alpha = np.linspace(0.0, 1.0, n, endpoint=True)
    mid = (1.0 - alpha) * a[-n:] + alpha * b[:n]
    return np.concatenate([a[:-n], mid, b[n:]])

def join_cost(a: np.ndarray, b: np.ndarray, sr: int = SR, ms: int = 25, nfft: int = 512) -> float:
    n = min(int(sr * ms / 1000.0), len(a), len(b))
    wa = np.hanning(n); wb = np.hanning(n)
    A = np.fft.rfft(a[-n:] * wa, n=nfft)
    B = np.fft.rfft(b[:n] * wb, n=nfft)
    La = 20.0 * np.log10(np.maximum(np.abs(A), 1e-8))
    Lb = 20.0 * np.log10(np.maximum(np.abs(B), 1e-8))
    spec_dist = float(np.mean((La - Lb) ** 2))
    rms_diff = abs(calc_rms(a[-n:]) - calc_rms(b[:n])) * 100.0
    return spec_dist + rms_diff

def target_cost(unit_path: str, target_dur: float = 0.38, target_rms: float = 0.13) -> float:
    x = load_unit(unit_path, SR)
    dur = len(x) / float(SR)
    r = calc_rms(x)
    return float(10.0 * ((dur - target_dur) ** 2) + 50.0 * ((r - target_rms) ** 2))

def viterbi_unit_selection(tokens: list, candidates: list = ["01", "02"], lambda_join: float = 1.0):
    N, K = len(tokens), len(candidates)
    dp = np.zeros((N, K), dtype=np.float32)
    parent = np.zeros((N, K), dtype=int)
    
    unit_cache = {}
    for tok in tokens:
        for c in candidates:
            p = os.path.join(UNITS_DIR, f"{VI_TO_FILENAME[tok]}_{c}.wav")
            unit_cache[(tok, c)] = match_rms(load_unit(p, SR))
            
    tok0 = tokens[0]
    for k, c in enumerate(candidates):
        p = os.path.join(UNITS_DIR, f"{VI_TO_FILENAME[tok0]}_{c}.wav")
        dp[0, k] = target_cost(p)
        parent[0, k] = -1
        
    for i in range(1, N):
        curr_tok, prev_tok = tokens[i], tokens[i - 1]
        for k_curr, c_curr in enumerate(candidates):
            curr_x = unit_cache[(curr_tok, c_curr)]
            p_curr = os.path.join(UNITS_DIR, f"{VI_TO_FILENAME[curr_tok]}_{c_curr}.wav")
            t_cost = target_cost(p_curr)
            best_cost, best_p = 1e9, 0
            for k_prev, c_prev in enumerate(candidates):
                prev_x = unit_cache[(prev_tok, c_prev)]
                j_cost = join_cost(prev_x, curr_x, SR)
                total_c = dp[i - 1, k_prev] + t_cost + lambda_join * j_cost
                if total_c < best_cost:
                    best_cost, best_p = total_c, k_prev
            dp[i, k_curr] = best_cost
            parent[i, k_curr] = best_p
            
    best_k = int(np.argmin(dp[N - 1, :]))
    min_total_cost = float(dp[N - 1, best_k])
    chosen = [None] * N
    curr = best_k
    for i in range(N - 1, -1, -1):
        chosen[i] = candidates[curr]
        curr = parent[i, curr]
    return chosen, min_total_cost

def synthesize_concatenative(tokens: list, candidate_seq: list, sr: int = SR, xfade_ms: int = 15):
    xs = [match_rms(load_unit(os.path.join(UNITS_DIR, f"{VI_TO_FILENAME[t]}_{c}.wav"), sr))
          for t, c in zip(tokens, candidate_seq)]
    y = xs[0]
    n_fade = int(sr * xfade_ms / 1000.0)
    boundaries = [0, len(y)]
    for x in xs[1:]:
        y = crossfade(y, x, n_fade)
        boundaries.append(len(y))
    return y, boundaries

chosen_cands, best_v_cost = viterbi_unit_selection(tokens_1)
print("Chuỗi Candidate tối ưu theo Viterbi DP:")
for t, c in zip(tokens_1, chosen_cands):
    print(f"  + {t:6s} -> candidate_{c}")
print(f"Tổng chi phí Viterbi Cost: {best_v_cost:.3f}")

# Tạo audio với các mức cross-fade: 0ms, 5ms, 15ms, 30ms
for ms in [0, 5, 15, 30]:
    y_test_xf, _ = synthesize_concatenative(tokens_1, chosen_cands, xfade_ms=ms)
    sf.write(os.path.join(OUTPUT_DIR, f"concat_xfade_{ms}ms.wav"), y_test_xf, SR)

y_best, b_best = synthesize_concatenative(tokens_1, chosen_cands, xfade_ms=15)
sf.write(os.path.join(OUTPUT_DIR, "best_concat.wav"), y_best, SR)
print("Đã lưu best_concat.wav (Viterbi + 15ms cross-fade).")
display(Audio(y_best, rate=SR))


In [ ]:
# Hiển thị đồ thị so sánh biên nối Raw vs Best (Figure 2)
fig, axes = plt.subplots(4, 1, figsize=(13, 9), sharex=False)
time_raw = np.arange(len(y_raw)) / float(SR)
time_best = np.arange(len(y_best)) / float(SR)

axes[0].plot(time_raw, y_raw, color='#d95f02', linewidth=0.7)
axes[0].set_title("(a) Raw Waveform (xfade=0ms, candidate cố định) - Click tại biên", fontweight='bold')
axes[0].set_ylabel("Biên độ"); axes[0].set_ylim(-1.0, 1.0)
for b in b_raw[1:-1]: axes[0].axvline(b / float(SR), color='black', linestyle='--', alpha=0.7)

D_raw = librosa.amplitude_to_db(np.abs(librosa.stft(y_raw, n_fft=512, hop_length=128)), ref=np.max)
axes[1].imshow(D_raw, aspect='auto', origin='lower', extent=[0, time_raw[-1], 0, SR/2], cmap='magma', vmin=-60, vmax=0)
axes[1].set_title("(b) Spectrogram Raw - Gián đoạn phổ tại các biên ghép", fontweight='bold')
axes[1].set_ylabel("Tần số (Hz)")
for b in b_raw[1:-1]: axes[1].axvline(b / float(SR), color='cyan', linestyle='--', alpha=0.8)

axes[2].plot(time_best, y_best, color='#1b9e77', linewidth=0.7)
axes[2].set_title("(c) Best Waveform (Viterbi + xfade=15ms) - Biên nối trơn mượt", fontweight='bold')
axes[2].set_ylabel("Biên độ"); axes[2].set_ylim(-1.0, 1.0)
for b in b_best[1:-1]: axes[2].axvline(b / float(SR), color='black', linestyle='--', alpha=0.7)

D_best = librosa.amplitude_to_db(np.abs(librosa.stft(y_best, n_fft=512, hop_length=128)), ref=np.max)
axes[3].imshow(D_best, aspect='auto', origin='lower', extent=[0, time_best[-1], 0, SR/2], cmap='magma', vmin=-60, vmax=0)
axes[3].set_title("(d) Spectrogram Best - Dải phổ chuyển tiếp tự nhiên, triệt tiêu artifact", fontweight='bold')
axes[3].set_xlabel("Thời gian (giây)"); axes[3].set_ylabel("Tần số (Hz)")
for b in b_best[1:-1]: axes[3].axvline(b / float(SR), color='cyan', linestyle='--', alpha=0.8)

plt.tight_layout()
plt.show()


## Phần D: Prosody & Mô hình Nguồn - Lọc (Source-Filter)
Mô hình toán học cơ bản của tiếng nói:
$$X(z) = E(z) \cdot H(z)$$
- $E(z)$: Nguồn kích thích thanh quản (Glottal excitation). Với tín hiệu hữu thanh (Voiced), $E(z)$ là chuỗi xung tuần hoàn với chu kỳ cơ bản $N_p$ mẫu $\Rightarrow F_0 = F_s / N_p$.
- $H(z)$: Bộ lọc đường thanh quản (Vocal-tract filter) quyết định vị trí các cộng hưởng âm học (Formants $F_1, F_2, F_3$).
- Khi điều chỉnh tốc độ nói (Duration Stretch) bằng kỹ thuật OLA/WSOLA (`librosa.effects.time_stretch`), tín hiệu thay đổi tốc độ nhưng giữ nguyên $F_0$ và đường bao phổ Formant.
- Khi dịch cao độ (Pitch Shift semitone) $F_{0, new} = F_0 \cdot 2^{\Delta s / 12}$, ta thay đổi tần số dao động của dây thanh mà không được phép thay đổi vị trí các formant để tránh méo giọng sóc chuột (chipmunk effect).


In [ ]:
# 1. Đoạn nguyên âm tuần hoàn và kiểm chứng F0 = Fs / Np
x_hai = load_unit(os.path.join(UNITS_DIR, "hai_01.wav"))
mid_idx = len(x_hai) // 2
vowel_seg = x_hai[mid_idx - 400: mid_idx + 400]

# Tự tương quan
r_corr = np.correlate(vowel_seg, vowel_seg, mode='full')
r_corr = r_corr[len(vowel_seg) - 1:]
r_corr = r_corr / (r_corr[0] + 1e-12)

min_lag, max_lag = int(SR / 400.0), int(SR / 80.0)
lag_peak = min_lag + np.argmax(r_corr[min_lag:max_lag])
f0_est = SR / float(lag_peak)
print(f"Kiểm chứng thực nghiệm: Chu kỳ pitch Np = {lag_peak} mẫu")
print(f"Tần số cơ bản F0 = Fs / Np = {SR} / {lag_peak} = {f0_est:.2f} Hz")

# 2. Thí nghiệm Duration Stretch (WSOLA)
y_speed_08 = librosa.effects.time_stretch(y_best, rate=1.25) # 0.8x duration
y_speed_12 = librosa.effects.time_stretch(y_best, rate=0.833) # 1.2x duration

# 3. Thí nghiệm Pitch Shift (-2, +2 semitones)
y_pitch_m2 = librosa.effects.pitch_shift(y_best, sr=SR, n_steps=-2)
y_pitch_p2 = librosa.effects.pitch_shift(y_best, sr=SR, n_steps=+2)

print("Thời lượng sau stretch: 0.8x =", len(y_speed_08)/SR, "s, 1.2x =", len(y_speed_12)/SR, "s")


## Phần E: Tổng hợp tiếng nói tham số dựa trên HMM
Khác với phương pháp ghép nối trực tiếp, HMM TTS hoạt động ở chế độ **Phát sinh (Generation)**:
1. Chuỗi ký hiệu/phone kích hoạt chuỗi trạng thái ẩn Left-to-Right.
2. Mỗi trạng thái mang phân bố xác suất Gaussian $\mathcal{N}(\mu_i, \Sigma_i)$ của vector đặc trưng âm học (F0, Formants $F_1, F_2, F_3$, Năng lượng).
3. **MLPG (Maximum Likelihood Parameter Generation)**: Áp dụng các ràng buộc đạo hàm động (Delta features $\Delta c_t = \frac{1}{2}(c_{t+1} - c_{t-1})$) để giải nghiệm Maximum-Likelihood, biến chuỗi giá trị nhảy bậc từ state thành quỹ đạo tham số âm học trơn tru, liên tục theo thời gian.
4. Bộ tổng hợp nguồn-lọc (Formant Vocoder) phát sinh dạng sóng từ các quỹ đạo tham số này.


In [ ]:
# Chạy tổng hợp tham số HMM
y_hmm, sr_hmm, tracks = synthesize_hmm(test_sent_1, dur_scale=1.0, f0_scale=1.0, sr=SR)
sf.write(os.path.join(OUTPUT_DIR, "hmm_tts.wav"), y_hmm, SR)
print(f"HMM Parametric TTS hoàn tất. Thời lượng: {len(y_hmm)/SR:.2f}s")
display(Audio(y_hmm, rate=SR))

# Vẽ quỹ đạo tham số HMM (F0, Formants, Energy)
fig, axes = plt.subplots(3, 1, figsize=(12, 7.5), sharex=True)
t = tracks["time"]

axes[0].plot(t, tracks["raw_f0"], ':', color='gray', label="Raw State Levels (Step function)")
axes[0].plot(t, tracks["f0"], color='#d95f02', linewidth=2.0, label="MLPG Smoothed F0 Contour")
axes[0].set_title("(a) F0 Contour (Hiệu ứng làm trơn của ràng buộc Delta MLPG)", fontweight='bold')
axes[0].set_ylabel("F0 (Hz)"); axes[0].grid(True, linestyle=':'); axes[0].legend(loc='upper right')

axes[1].plot(t, tracks["f1"], color='#1b9e77', linewidth=1.8, label="Formant F1")
axes[1].plot(t, tracks["f2"], color='#7570b3', linewidth=1.8, label="Formant F2")
axes[1].plot(t, tracks["f3"], color='#e7298a', linewidth=1.8, label="Formant F3")
axes[1].set_title("(b) Quỹ đạo các Formant âm học (F1, F2, F3)", fontweight='bold')
axes[1].set_ylabel("Tần số (Hz)"); axes[1].grid(True, linestyle=':'); axes[1].legend(loc='upper right')

axes[2].fill_between(t, 0, tracks["energy"], color='#e6ab02', alpha=0.4)
axes[2].plot(t, tracks["energy"], color='#e6ab02', linewidth=1.5, label="Energy Gain")
axes[2].set_title("(c) Đường bao năng lượng (Gain/Energy Contour)", fontweight='bold')
axes[2].set_xlabel("Thời gian (giây)"); axes[2].set_ylabel("Biên độ"); axes[2].grid(True, linestyle=':')

plt.tight_layout()
plt.show()


## Phần F: Neural TTS học sâu bằng mô hình Pretrained
Kiến trúc TTS hiện đại (End-to-End Neural TTS):
- **Front-end / Text Processor**: Phân tích từ vựng, âm tiết, ngữ điệu.
- **Acoustic Model**: Dự đoán biểu diễn âm học trung gian chất lượng cao (Mel-spectrogram 80 băng tần) từ biểu diễn văn bản.
- **Neural Vocoder**: Tổng hợp trực tiếp waveform từ Mel-spectrogram với độ tự nhiên cao vượt bậc, bảo toàn toàn bộ harmonic và phase.


In [ ]:
# Chạy inference mô hình Neural TTS
y_nn, sr_nn, extra = synthesize(test_sent_1, sr=SR)
sf.write(os.path.join(OUTPUT_DIR, "neural_tts.wav"), y_nn, SR)

print(f"Neural TTS Inference thành công!")
print(f"- Văn bản: '{extra['text']}'")
print(f"- Độ trễ (Latency): {extra['latency_ms']:.1f} ms")
print(f"- Thời lượng audio: {extra['duration_sec']:.2f} s")
print(f"- Real-Time Factor (RTF): {extra['rtf']:.3f}")
display(Audio(y_nn, rate=SR))


## Phần G: Đánh giá & So sánh Đối đầu 3 Phương pháp
So sánh đối đầu giữa 3 trường phái tổng hợp tiếng nói:
1. **Concatenative TTS (Ghép nối đơn vị)**
2. **HMM-based Statistical Parametric TTS (Tổng hợp tham số)**
3. **Deep Learning Neural TTS (Mạng nơ-ron học sâu)**


In [ ]:
# Đọc kết quả đánh giá chủ quan MOS từ ratings.csv
df_ratings = pd.read_csv("ratings.csv")
display(df_ratings)

# Thống kê trung bình theo từng phương pháp
summary_mos = df_ratings.groupby("method")[["intelligibility", "naturalness", "continuity", "latency_ms", "footprint_mb"]].mean()
print("=== BẢNG TỔNG KẾT SO SÁNH 3 HƯỚNG TỔNG HỢP TIẾNG NÓI ===")
display(summary_mos.round(2))


## Kết luận bài Lab 4
1. **Concatenative TTS**: Tốc độ xử lý cực nhanh (latency < 15ms), độ rõ (intelligibility) cao trong miền từ vựng đóng, nhưng dễ lộ vết nối tại biên nếu không có cơ chế Unit Selection tối ưu và Cross-fade phù hợp.
2. **HMM Parametric TTS**: Dung lượng lưu trữ siêu nhỏ (< 2MB footprint), điều khiển prosody (F0, duration) cực kỳ linh hoạt, tính liền mạch cao, nhưng chất lượng giọng còn mang âm sắc cơ giới/robotic đặc trưng do bộ vocoder nguồn lọc thô sơ.
3. **Neural TTS**: Đạt chất lượng vượt trội về độ tự nhiên (MOS > 4.8), nhịp điệu và ngữ điệu hoàn hảo, là tiêu chuẩn công nghiệp hiện nay, đánh đổi lại độ trễ xử lý và tài nguyên tính toán lớn hơn.
